CELDA 1 — Imports y carga

In [22]:
# ─── Preprocesamiento · p7_g1_multiclase ──────────────────────────────────────
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import joblib, os

df = pd.read_csv('../data/processed/ObesityDataSet_clean.csv')
print(f"✅ Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head(3)

✅ Dataset cargado: 2087 filas x 17 columnas


,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,Female,21.0,1.62,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight
1,Female,21.0,1.52,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight
2,Male,23.0,1.80,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight


CELDA 2 — Identificar columnas por tipo

In [23]:
# ─── Clasificar columnas ───────────────────────────────────────────────────────
TARGET = 'NObeyesdad'

# Binarias (yes/no) → LabelEncoder o map directo
cols_binarias = ['family_history_with_overweight', 'FAVC', 'SMOKE', 'SCC']

# Ordinales con orden natural → OrdinalEncoder con categorías explícitas
cols_ordinales = {
    'CAEC':  ['no', 'Sometimes', 'Frequently', 'Always'],
    'CALC':  ['no', 'Sometimes', 'Frequently', 'Always'],
    'MTRANS': ['Walking', 'Bike', 'Motorbike', 'Public_Transportation', 'Automobile'],
}

# Nominal con pocas categorías → get_dummies / OneHotEncoder
cols_nominales = ['Gender']

# Numéricas continuas
cols_numericas = ['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']

print("Binarias:  ", cols_binarias)
print("Ordinales: ", list(cols_ordinales.keys()))
print("Nominales: ", cols_nominales)
print("Numéricas: ", cols_numericas)

Binarias:   ['family_history_with_overweight', 'FAVC', 'SMOKE', 'SCC']
Ordinales:  ['CAEC', 'CALC', 'MTRANS']
Nominales:  ['Gender']
Numéricas:  ['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']


CELDA 3 — Encoding manual de binarias y target

In [24]:
# ─── Encoding binarias (yes=1, no=0) ──────────────────────────────────────────
binary_map = {'yes': 1, 'no': 0}
for col in cols_binarias:
    df[col] = df[col].map(binary_map)

# ─── Encoding del target con orden clínico real ───────────────────────────────
# LabelEncoder ordena alfabéticamente → NO preserva la progresión de obesidad
# Solución: mapeo manual con orden clínico correcto

orden_clinico = [
    'Insufficient_Weight',   # 0
    'Normal_Weight',         # 1
    'Overweight_Level_I',    # 2
    'Overweight_Level_II',   # 3
    'Obesity_Type_I',        # 4
    'Obesity_Type_II',       # 5
    'Obesity_Type_III',      # 6
]

target_map = {clase: i for i, clase in enumerate(orden_clinico)}
df['target_encoded'] = df[TARGET].map(target_map)

# Verificación: no debe haber NaN tras el mapeo
assert df['target_encoded'].isna().sum() == 0, "❌ Hay valores no mapeados en target"
print(f"\n✅ Mapeo correcto — {df['target_encoded'].nunique()} clases, sin NaN")
print(df[[TARGET, 'target_encoded']].drop_duplicates().sort_values('target_encoded'))

# Encoding del target
le_target = LabelEncoder()
df['target_encoded'] = le_target.fit_transform(df[TARGET])
print("Clases del target:")
for i, cls in enumerate(le_target.classes_):
    print(f"  {i} → {cls}")


✅ Mapeo correcto — 7 clases, sin NaN
              NObeyesdad  target_encoded
59   Insufficient_Weight               0
0          Normal_Weight               1
3     Overweight_Level_I               2
4    Overweight_Level_II               3
10        Obesity_Type_I               4
68       Obesity_Type_II               5
197     Obesity_Type_III               6
Clases del target:
  0 → Insufficient_Weight
  1 → Normal_Weight
  2 → Obesity_Type_I
  3 → Obesity_Type_II
  4 → Obesity_Type_III
  5 → Overweight_Level_I
  6 → Overweight_Level_II


In [30]:
df.head()

,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad,target_encoded,Gender_Male
0,21.0,1.62,64.0,1,0,2.0,3.0,1.0,0,2.0,0,0.0,1.0,0.0,3.0,Normal_Weight,1,False
1,21.0,1.52,56.0,1,0,3.0,3.0,1.0,1,3.0,1,3.0,0.0,1.0,3.0,Normal_Weight,1,False
2,23.0,1.80,77.0,1,0,2.0,3.0,1.0,0,2.0,0,2.0,1.0,2.0,3.0,Normal_Weight,1,True
3,27.0,1.80,87.0,0,0,3.0,3.0,1.0,0,2.0,0,2.0,0.0,2.0,0.0,Overweight_Level_I,5,True
4,22.0,1.78,89.8,0,0,2.0,1.0,1.0,0,2.0,0,0.0,0.0,1.0,3.0,Overweight_Level_II,6,True


CELDA 4 — Encoding ordinales y nominales

In [25]:
# ─── Ordinales ────────────────────────────────────────────────────────────────
for col, orden in cols_ordinales.items():
    oe = OrdinalEncoder(categories=[orden])
    df[col] = oe.fit_transform(df[[col]])

# ─── Nominales (Gender) → One-Hot ─────────────────────────────────────────────
df = pd.get_dummies(df, columns=cols_nominales, drop_first=True)  # Gender_Male = 1

print("✅ Encoding completado")
print(df.dtypes)

✅ Encoding completado
Age                               float64
Height                            float64
Weight                            float64
family_history_with_overweight      int64
FAVC                                int64
FCVC                              float64
NCP                               float64
CAEC                              float64
SMOKE                               int64
CH2O                              float64
SCC                                 int64
FAF                               float64
TUE                               float64
CALC                              float64
MTRANS                            float64
NObeyesdad                            str
target_encoded                      int64
Gender_Male                          bool
dtype: object


CELDA 5 — Split train/test

In [26]:
# ─── Split estratificado (mantiene proporción de clases) ──────────────────────
X = df.drop(columns=[TARGET, 'target_encoded'])
y = df['target_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # ← crítico en multiclase
)

print(f"Train: {X_train.shape[0]} filas | Test: {X_test.shape[0]} filas")
print(f"\nDistribución y_train:\n{y_train.value_counts().sort_index()}")
print(f"\nDistribución y_test:\n{y_test.value_counts().sort_index()}")

Train: 1669 filas | Test: 418 filas

Distribución y_train:
target_encoded
0    214
1    225
2    281
3    237
4    259
5    221
6    232
Name: count, dtype: int64

Distribución y_test:
target_encoded
0    53
1    57
2    70
3    60
4    65
5    55
6    58
Name: count, dtype: int64


CELDA 6 — Escalado (solo para KNN y SVM)

In [27]:
# ─── Scaler FIT solo sobre train, transform sobre ambos ───────────────────────
# ⚠️  Árboles y XGBoost NO necesitan escalado → usarán X_train / X_test directos
# KNN y SVM → usarán X_train_scaled / X_test_scaled

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),      # ← transform, nunca fit_transform aquí
    columns=X_test.columns,
    index=X_test.index
)

print("✅ Escalado completado")
print(X_train_scaled.describe().round(2))

✅ Escalado completado
           Age   Height   Weight  family_history_with_overweight     FAVC  \
count  1669.00  1669.00  1669.00                         1669.00  1669.00   
mean     -0.00     0.00    -0.00                            0.00    -0.00   
std       1.00     1.00     1.00                            1.00     1.00   
min      -1.47    -2.62    -1.83                           -2.17    -2.71   
25%      -0.70    -0.76    -0.83                            0.46     0.37   
50%      -0.23    -0.01    -0.13                            0.46     0.37   
75%       0.27     0.72     0.81                            0.46     0.37   
max       5.81     2.99     2.82                            0.46     0.37   

          FCVC      NCP     CAEC    SMOKE     CH2O      SCC      FAF      TUE  \
count  1669.00  1669.00  1669.00  1669.00  1669.00  1669.00  1669.00  1669.00   
mean     -0.00     0.00     0.00     0.00     0.00     0.00     0.00    -0.00   
std       1.00     1.00     1.00     1.00

CELDA 7 — Guardar artefactos

In [28]:
# ─── Guardar todo lo necesario para el notebook de modelado ───────────────────
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# Datos
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
X_train_scaled.to_csv('../data/processed/X_train_scaled.csv', index=False)
X_test_scaled.to_csv('../data/processed/X_test_scaled.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

# Objetos sklearn (para invertir predicciones luego)
joblib.dump(scaler,    '../models/scaler.pkl')
joblib.dump(le_target, '../models/label_encoder_target.pkl')

print("✅ Artefactos guardados:")
print("   · X_train / X_test (para árboles y XGBoost)")
print("   · X_train_scaled / X_test_scaled (para KNN y SVM)")
print("   · scaler.pkl  · label_encoder_target.pkl")

✅ Artefactos guardados:
   · X_train / X_test (para árboles y XGBoost)
   · X_train_scaled / X_test_scaled (para KNN y SVM)
   · scaler.pkl  · label_encoder_target.pkl


Dos puntos clave a tener en cuenta:
Data leakage: el scaler.fit_transform se hace solo sobre X_train. El X_test solo recibe .transform(). Si lo hicieras al revés el modelo vería información del test durante el entrenamiento.
Qué datos usa cada modelo en el notebook de modelado:
ModeloX que usaDecision Tree / Random Forest / XGBoostX_train, X_testKNN, SVMX_train_scaled, X_test_scaled